# Section 4 – Model 2: predict `is_legendary` (classification)
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Your ML background: Logistic Regression vs Random Forest, imbalanced classes |
| Estimated time | 60 minutes |
| Produces | `models/classifier.joblib` and `models/classifier_meta.json` |

**Goal:** Train and compare a Logistic Regression and a Random Forest classifier on a rare class. Learn why accuracy lies and why precision, recall and probabilities matter.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## The question
> Given a character's **8 stats**, is it a **legendary / mythical** character? And with what probability?

On the card the **probability** decides the rarity border:

| Probability of being legendary | Rarity tier | Card border |
|---|---|---|
| below 20% | common | grey |
| 20% to 50% | rare | blue |
| 50% or more | legendary | gold |

| Model | Idea |
|---|---|
| **Logistic Regression** | a linear score squeezed into a probability between 0 and 1 |
| **Random Forest** | many trees vote; the share of "yes" votes is the probability |

The big difficulty: legendary characters are **rare** (about 1 in 10). This is an *imbalanced* problem.

## Step 1 – Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

clean = pd.read_csv(DATA / "pokemon_clean.csv")
info = json.loads((DATA / "feature_info.json").read_text(encoding="utf-8"))
FEATURES = info["features"]
TARGET = info["targets"]["classification"]
print("target:", TARGET)
print(clean[TARGET].value_counts().rename({0: "common", 1: "legendary/mythical"}))

## Step 2 – Stratified split
With a rare class, a random split could leave almost no legendary characters in the test set.
`stratify=y` keeps the **same proportion** of each class in both parts.

🟦 **Worked sample** (toy data: 8% of heroes are "bosses"):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)
power = np.concatenate([rng.normal(40, 10, 460), rng.normal(75, 10, 40)])
is_boss = np.array([0] * 460 + [1] * 40)

p_train, p_test, b_train, b_test = train_test_split(power.reshape(-1, 1), is_boss, test_size=0.2,
                                                    random_state=42, stratify=is_boss)
print("boss share in train:", b_train.mean().round(3), "| in test:", b_test.mean().round(3))
toy_clf = LogisticRegression().fit(p_train, b_train)
print("probability that power=80 is a boss:", toy_clf.predict_proba([[80]])[0, 1].round(3))

In [ ]:
# TODO: X and y from `clean`
...  # <- write your code here

# TODO: stratified 80/20 split with random_state=42
# HINT: train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
...  # <- write your code here

print("legendary share  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))
check("same class share in both parts", abs(y_train.mean() - y_test.mean()) < 0.01)
check("test set has at least 10 positives", int(y_test.sum()) >= 10, int(y_test.sum()))

## Step 3 – The accuracy trap
Predict "not legendary" for everybody and you are right ~90% of the time... and you never find a single legendary.
That is why we also use:

| Metric | Question it answers |
|---|---|
| **Precision** | of the ones I called legendary, how many really are? |
| **Recall** | of the real legendaries, how many did I find? |
| **F1** | one number balancing precision and recall |
| **ROC-AUC** | how well the probabilities rank legendaries above commons (0.5 = random, 1 = perfect) |

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def evaluate_clf(model, X_eval, y_eval, threshold=0.5):
    """Metrics of a classifier. Uses predict_proba and the given probability threshold."""
    # TODO: probabilities of class 1, then 0/1 predictions with the threshold, then the five metrics
    # HINT: proba = model.predict_proba(X_eval)[:, 1]; pred = (proba >= threshold).astype(int)
    ...  # <- write your code here
    return {k: round(float(v), 3) for k, v in out.items()}


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
results = {"baseline (always 'common')": evaluate_clf(baseline, X_test, y_test)}
print(results)
check("baseline accuracy looks great (> 0.85)...", results["baseline (always 'common')"]["accuracy"] > 0.85)
check("...but its recall is 0", results["baseline (always 'common')"]["recall"] == 0)

## Step 4 – Model A: Logistic Regression
Two details matter here:
- **Scaling** (`StandardScaler`), as with any linear model.
- `class_weight="balanced"`: mistakes on the rare class cost more, so the model does not ignore it.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TODO: pipeline: StandardScaler + LogisticRegression(class_weight="balanced", max_iter=1000); fit on TRAIN only
# HINT: make_pipeline(StandardScaler(), LogisticRegression(...))
...  # <- write your code here

results["Logistic Regression"] = evaluate_clf(log, X_test, y_test)
print(results["Logistic Regression"])
odds = pd.Series(log[-1].coef_[0], index=FEATURES).sort_values()
print("\nCoefficients (positive = pushes towards legendary):")
print(odds.round(2))
check("recall is clearly above the baseline", results["Logistic Regression"]["recall"] > 0.5,
      results["Logistic Regression"]["recall"])

## Step 5 – Model B: Random Forest classifier
`class_weight="balanced_subsample"` re-balances inside every tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: RandomForestClassifier: 300 trees, class_weight="balanced_subsample", random_state=42, n_jobs=-1; fit on TRAIN only
...  # <- write your code here

results["Random Forest"] = evaluate_clf(forest, X_test, y_test)
print(results["Random Forest"])
pd.Series(forest.feature_importances_, index=FEATURES).sort_values().plot.barh(
    color="#0066cc", figsize=(6, 3.5), title="Random Forest feature importance")
plt.show()
check("forest finds some legendaries (recall > 0)", results["Random Forest"]["recall"] > 0,
      results["Random Forest"]["recall"])

**Compare Step 4 and Step 5 before moving on.** Which model has the higher *precision* and which the higher *recall* at the 0.5 threshold?
A forest is usually more **cautious**: it says "legendary" only when many trees agree. Write down why that could be good for one use
(avoid false gold cards) and bad for another (find every legendary). Step 7 shows how the threshold changes this.

## Step 6 – The confusion matrix: where exactly are the mistakes?
```
                predicted common   predicted legendary
actual common        TN                   FP   (a common card with a gold border)
actual legendary     FN                   TP
```
**FN** = a legendary card shown as common. **FP** = a common card shown as legendary. Which one is worse for a game?

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: confusion matrices of both models on the test set (rows = actual, columns = predicted)
# HINT: confusion_matrix(y_test, model.predict(X_test))
...  # <- write your code here

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, name, cm in zip(axes, ["Logistic Regression", "Random Forest"], [cm_log, cm_rf]):
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, v, ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1], ["common", "legendary"])
    ax.set_yticks([0, 1], ["common", "legendary"])
    ax.set_title(name)
    ax.set_xlabel("predicted")
axes[0].set_ylabel("actual")
plt.tight_layout()
plt.show()
print(classification_report(y_test, forest.predict(X_test), target_names=["common", "legendary"]))
check("matrix totals match the test size", cm_log.sum() == len(y_test) and cm_rf.sum() == len(y_test))

## Step 7 – Probabilities and thresholds
The default "legendary if probability ≥ 0.5" is a choice, not a law. Lowering the threshold finds more legendaries
(higher recall) but also more false alarms (lower precision). The card uses the **probability itself** (three tiers), so this
trade-off is visible to the player.

In [ ]:
# TODO: for thresholds 0.2, 0.35, 0.5 and 0.7 compute the metrics of the Random Forest and keep them in a DataFrame `tradeoff`
# HINT: evaluate_clf(forest, X_test, y_test, threshold=t)
...  # <- write your code here

print(tradeoff)
check("recall never increases when the threshold goes up", tradeoff["recall"].is_monotonic_decreasing)

## Step 8 – Cross-validation (important with so few positives)
Our test set has only ~20 legendaries: one lucky case changes the metrics a lot. **Stratified 5-fold CV** is more stable.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: CV F1 score of both models on the whole X, y using `skf`
# HINT: cross_val_score(model, X, y, cv=skf, scoring="f1")
...  # <- write your code here

print(f"Logistic Regression CV F1: {cv_log.mean():.3f} ± {cv_log.std():.3f}")
print(f"Random Forest       CV F1: {cv_rf.mean():.3f} ± {cv_rf.std():.3f}")
results["Logistic Regression"]["cv_f1"] = round(float(cv_log.mean()), 3)
results["Random Forest"]["cv_f1"] = round(float(cv_rf.mean()), 3)

## Step 9 – Choose and save
Rule: pick the best **CV F1**; if the two are within 0.02 of each other, prefer Logistic Regression (simpler).
The saved model must expose `predict_proba`: the API needs probabilities, not just yes/no.

In [ ]:
import joblib
import sklearn
from datetime import datetime

candidates = {"Logistic Regression": log, "Random Forest": forest}

# TODO: best_name by the rule above
# HINT: compare results[name]["cv_f1"]
...  # <- write your code here

joblib.dump(candidates[best_name], MODELS / "classifier.joblib")
meta = {
    "name": best_name, "target": TARGET, "features": FEATURES,
    "metrics": results[best_name], "all_results": results,
    "positive_rate": round(float(y.mean()), 4), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "trained_at": datetime.now().isoformat(timespec="seconds"), "sklearn_version": sklearn.__version__,
}
(MODELS / "classifier_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved:", best_name, results[best_name])

In [ ]:
loaded = joblib.load(MODELS / "classifier.joblib")
cases = pd.DataFrame([
    {"name": "mewtwo-like", "hp": 106, "attack": 110, "defense": 90, "special_attack": 154, "special_defense": 90, "speed": 130, "height": 20, "weight": 1220},
    {"name": "magikarp-like", "hp": 20, "attack": 10, "defense": 55, "special_attack": 15, "special_defense": 20, "speed": 80, "height": 9, "weight": 100},
])
cases["P(legendary)"] = loaded.predict_proba(cases[FEATURES])[:, 1].round(3)
print(cases[["name", "P(legendary)"]])

check("classifier file saved", (MODELS / "classifier.joblib").exists())
check("supports predict_proba", hasattr(loaded, "predict_proba"))
check("mewtwo-like is far more legendary than magikarp-like", cases["P(legendary)"][0] > cases["P(legendary)"][1] + 0.3)

## Questions
1. The baseline had ~90% accuracy and 0 recall. Explain in one sentence why accuracy is a bad metric here.
2. Which error is worse for a game card: FN or FP? How would you change the threshold because of that?
3. Why do we use `stratify=y` and a stratified CV?
4. Which model did you save and why? Would your answer change with 10x more data?

### Extra challenge
Remove `class_weight` from both models and compare recall. What happens to the probabilities?

**Next:** Section 5 – package both models and validate input with Pydantic.